In [ ]:
import json
from pathlib import Path

EVID_DIR = Path("/home/jupyter/evidence")
files = sorted(p for p in EVID_DIR.glob("*.json") if p.stem.isdigit())
print("evidence files:", len(files))

def show(x, indent=0, max_keys=6):
    pad = "  " * indent
    if isinstance(x, dict):
        keys = list(x)
        print(f"{pad}dict ({len(keys)} keys)")
        for k in keys[:max_keys]:
            print(f"{pad}  [{k!r}]")
            show(x[k], indent + 2)
        if len(keys) > max_keys:
            print(f"{pad}  ... {len(keys) - max_keys} more keys")
    elif isinstance(x, list):
        print(f"{pad}list (len {len(x)})")
        if x:
            show(x[0], indent + 1)
    elif isinstance(x, str):
        print(f"{pad}str (len {len(x)})")
    else:
        print(f"{pad}{type(x).__name__}")

show(json.load(open(files[0], encoding="utf-8")))

In [ ]:
ev = json.load(open(files[0], encoding="utf-8"))
print(ev["specialists"]["Cardiology"]["kb_groups"][0])
print(ev["model2"]["kb_groups"][0])

In [ ]:
import json, re
from pathlib import Path
from transformers import AutoTokenizer

# ---------------- settings (fixed; do not change after seeing results)
MODEL_ID       = "bbarn4/medgemma-27b-text-it-GPTQ"
SERVER_URL     = "http://127.0.0.1:8000/v1"
EVID_DIR       = Path("evidence")
OUT_DIR        = Path("model3_outputs")
PRED_FILE      = Path("predictions_model3_27b.json")
MAX_NEW_TOKENS     = 600       # same as Model 1
REPETITION_PENALTY = 1.05      # same as Model 1
CONTEXT_WINDOW     = 40_000    # vLLM --max-model-len
SPECIALISTS = ["Cardiology", "Pulmonology", "Nephrology", "Infectious Diseases"]
SPEC_MAX, FINAL_MAX = 10, 15

(OUT_DIR / "specialists").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "moderator").mkdir(parents=True, exist_ok=True)

tok  = AutoTokenizer.from_pretrained(MODEL_ID)
ntok = lambda s: len(tok.encode(s, add_special_tokens=False))

def prompt_tokens(p):        # exact count including the chat template
    return ntok(tok.apply_chat_template([{"role": "user", "content": p}],
                                        tokenize=False, add_generation_prompt=True))

EV = {p.stem: json.load(open(p, encoding="utf-8"))
      for p in sorted(EVID_DIR.glob("*.json")) if p.stem.isdigit()}
assert len(EV) == 60, len(EV)

def case_key(ev):            # same key format as Model 1 predictions
    return f"patient_{ev['subject_id']}_{ev['hadm_id']}.txt"

# ---------------- evidence rendering (identical to 7-Shared_Evidence)
def fmt_patient(ev, rows):
    parts, cur = [], None
    for r in sorted(rows):
        c = ev["patient_chunks"][str(r)]
        if c["section"] != cur:
            parts.append(f"\n{c['section']}"); cur = c["section"]
        parts.append(c["text"].strip())
    return "\n".join(parts).strip()

def fmt_kb(ev, groups):
    parts = []
    for header, keys in groups:
        parts.append(f"\n{header}")
        parts += [f"[{ev['kb_chunks'][k]['source_file']}]\n{ev['kb_chunks'][k]['text'].strip()}" for k in keys]
    return "\n\n".join(parts).strip()

# ---------------- prompts (same structure as Model 1: evidence first, instructions after)
SPECIALIST_INSTRUCTIONS = """
You are a {spec} specialist in a multidisciplinary team reviewing a single hospital admission.
The patient evidence consists of excerpts selected from the record of this admission.
The guideline evidence consists of excerpts from clinical practice guidelines; it is reference
knowledge only and is not information about this patient. The candidate conditions in the
guideline headings were used only to retrieve evidence; do not assume the patient has them.

Using only the information in the patient evidence, predict the discharge diagnoses within {spec}.

List the diagnoses as one list, most important first. Include {spec} conditions that were
actively diagnosed or treated during this admission (including new problems and
complications that developed during the stay, as shown by lab trends, medications,
procedures and ICU events) and important chronic {spec} conditions.
List each diagnosis once. At most {n} diagnoses.
If the patient evidence supports no {spec} diagnosis, output only: None

Use only information supported by the patient evidence.
Do not invent unsupported clinical information.
Output only the numbered list and nothing else.
""".strip()

MODERATOR_INSTRUCTIONS = """
You are the chair of a multidisciplinary team reviewing a single hospital admission.
Four specialists (Cardiology, Pulmonology, Nephrology, Infectious Diseases) have each listed
the discharge diagnoses within their specialty, based on excerpts from the record of this admission.

Produce the final list of discharge diagnoses for this admission:
- Use only diagnoses proposed by at least one specialist. Do not add new diagnoses.
- Merge diagnoses that are the same or overlap into one entry, using the most specific
  wording supported by the patient evidence.
- Where specialists disagree, use the patient evidence to decide. Leave out a diagnosis
  that the patient evidence does not support.
- Order the list most important first: the main reason for this admission first, then other
  conditions actively diagnosed or treated during this admission, then important chronic conditions.
List each diagnosis once. At most {n} diagnoses.

Do not invent unsupported clinical information.
Output only the numbered list and nothing else.
""".strip()

def specialist_prompt(ev, spec):
    sp = ev["specialists"][spec]
    return ("PATIENT EVIDENCE:\n" + fmt_patient(ev, sp["patient_rows"])
            + "\n\n---\n\nGUIDELINE EVIDENCE:\n" + fmt_kb(ev, sp["kb_groups"])
            + "\n\n---\n\nThe patient evidence and guideline evidence are above.\n\n"
            + SPECIALIST_INSTRUCTIONS.format(spec=spec, n=SPEC_MAX))

def fmt_opinions(spec_lists):
    out = []
    for s in SPECIALISTS:
        dx = spec_lists[s][:SPEC_MAX]
        out.append(f"{s}:\n" + ("\n".join(f"{i}. {d}" for i, d in enumerate(dx, 1)) if dx else "None"))
    return "\n\n".join(out)

def moderator_prompt(ev, spec_lists):
    return ("PATIENT EVIDENCE:\n" + fmt_patient(ev, ev["model2"]["patient_rows"])
            + "\n\n---\n\nSPECIALIST OPINIONS:\n" + fmt_opinions(spec_lists)
            + "\n\n---\n\nThe patient evidence and specialist opinions are above.\n\n"
            + MODERATOR_INSTRUCTIONS.format(n=FINAL_MAX))

# ---------------- parser (identical to Model 1)
LIST_RE = re.compile(r"^\s*(?:\d+[.)]|[-*\u2022])\s*(.+?)\s*$")
def parse_diagnoses(raw):
    return [m.group(1) for line in raw.splitlines() if (m := LIST_RE.match(line))]

print("Cases loaded:", len(EV))

In [ ]:
# Checking if a median or maximum length case can fit in the text window of MedGemma 27B
import pandas as pd

log = pd.read_csv("evidence_token_log.csv", dtype={"hadm_id": str}).set_index(["hadm_id", "view"])
dummy = {s: ["placeholder diagnosis with a fairly long descriptive name"] * SPEC_MAX for s in SPECIALISTS}

mismatch, spec_tok, mod_tok = 0, [], []
for h, ev in EV.items():
    for s in SPECIALISTS:
        sp = ev["specialists"][s]
        ref = log.loc[(h, s)]
        mismatch += (ntok(fmt_patient(ev, sp["patient_rows"])) != ref.patient_tokens) \
                  or (ntok(fmt_kb(ev, sp["kb_groups"])) != ref.kb_tokens)
        spec_tok.append(prompt_tokens(specialist_prompt(ev, s)))
    ref = log.loc[(h, "Model2")]
    mismatch += ntok(fmt_patient(ev, ev["model2"]["patient_rows"])) != ref.patient_tokens
    mod_tok.append(prompt_tokens(moderator_prompt(ev, dummy)))

print("Evidence text mismatches vs token log (expect 0):", mismatch)
for name, t in (("Specialist", spec_tok), ("Moderator (worst-case lists)", mod_tok)):
    print(f"{name:<30} prompts={len(t)}  median={int(pd.Series(t).median()):,}  max={max(t):,}"
          f"  max+output={max(t)+MAX_NEW_TOKENS:,} / {CONTEXT_WINDOW:,}"
          f"  -> {'OK' if max(t)+MAX_NEW_TOKENS <= CONTEXT_WINDOW else 'TOO LONG'}")

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

client = OpenAI(base_url=SERVER_URL, api_key="x")
served = [m.id for m in client.models.list().data]
assert MODEL_ID in served, f"Server is serving {served}"
print("Server OK")

def spec_file(h, s): return OUT_DIR / "specialists" / f"{h}__{s.replace(' ', '_')}.json"
def mod_file(h):     return OUT_DIR / "moderator" / f"{h}.json"

def save(path, obj):                       # write to temp file, then rename (no half-written files)
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    tmp.replace(path)

def generate(prompt):                      # identical decoding to Model 1
    t0 = time.time()
    r = client.chat.completions.create(
        model=MODEL_ID, temperature=0, max_tokens=MAX_NEW_TOKENS,
        messages=[{"role": "user", "content": prompt}],
        extra_body={"repetition_penalty": REPETITION_PENALTY})
    raw = r.choices[0].message.content
    return {"raw": raw, "diagnoses": parse_diagnoses(raw), "finish": r.choices[0].finish_reason,
            "prompt_tokens": r.usage.prompt_tokens, "completion_tokens": r.usage.completion_tokens,
            "seconds": round(time.time() - t0, 1)}

def clean_list(dx):                        # drop "None" items, cap at SPEC_MAX
    dx = [d for d in dx if d.strip().lower().rstrip(".") != "none"]
    return dx[:SPEC_MAX], max(0, len(dx) - SPEC_MAX)

def run_specialist(h, s):
    out = generate(specialist_prompt(EV[h], s))
    out.update(hadm_id=h, specialist=s)
    save(spec_file(h, s), out)
    return out

def run_moderator(h):
    lists, cut = {}, 0
    for s in SPECIALISTS:
        lists[s], c = clean_list(json.load(open(spec_file(h, s), encoding="utf-8"))["diagnoses"])
        cut += c
    out = generate(moderator_prompt(EV[h], lists))
    out.update(hadm_id=h, specialist_lists=lists, specialist_items_cut=cut)
    save(mod_file(h), out)
    return out

def run_stage1(hadms, workers=4):
    idx = {h: i for i, h in enumerate(hadms, 1)}
    todo = [(h, s) for h in hadms for s in SPECIALISTS if not spec_file(h, s).exists()]
    print(f"Stage 1 | to do: {len(todo)} | already done: {4*len(hadms) - len(todo)}")
    t0 = time.time()
    with ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(run_specialist, h, s): (h, s) for h, s in todo}
        for n, f in enumerate(as_completed(futs), 1):
            h, s = futs[f]
            try:
                o = f.result()
                print(f"  case {idx[h]:>2} {s:<20} prompt={o['prompt_tokens']:>6,} out={o['completion_tokens']:>4} "
                      f"sec={o['seconds']:>6} finish={o['finish']:<6} n_dx={len(o['diagnoses'])}")
            except Exception as e:
                print(f"  case {idx[h]:>2} {s:<20} FAILED {type(e).__name__}: {str(e)[:120]}")
    print(f"Stage 1 wall time: {(time.time()-t0)/60:.1f} min")

def norm(d): return re.sub(r"[^a-z0-9 ]", "", d.lower()).strip()

def run_stage2(hadms):
    idx = {h: i for i, h in enumerate(hadms, 1)}
    ready = [h for h in hadms if all(spec_file(h, s).exists() for s in SPECIALISTS)]
    todo = [h for h in ready if not mod_file(h).exists()]
    print(f"Stage 2 | ready: {len(ready)} | to do: {len(todo)}")
    for h in todo:
        try:
            o = run_moderator(h)
        except Exception as e:
            print(f"  case {idx[h]:>2} moderator FAILED {type(e).__name__}: {str(e)[:120]}"); continue
        pool = {norm(d) for l in o["specialist_lists"].values() for d in l}
        exact = sum(norm(d) in pool for d in o["diagnoses"])
        dups = len(o["diagnoses"]) - len({norm(d) for d in o["diagnoses"]})
        print(f"  case {idx[h]:>2} moderator prompt={o['prompt_tokens']:>6,} sec={o['seconds']:>6} "
              f"finish={o['finish']:<6} n_in={len(pool)} n_out={len(o['diagnoses'])} "
              f"exact_match_to_specialist={exact} duplicates={dups} cut_items={o['specialist_items_cut']}")

In [ ]:
L = log.reset_index()
tot = L[L["view"] != "Model2"].groupby("hadm_id")["total_tokens"].max()
PILOT = [tot.idxmax(), (tot - tot.median()).abs().idxmin()]

run_stage1(PILOT)
run_stage2(PILOT)

In [ ]:
for i, h in enumerate(PILOT, 1):
    o = json.load(open(mod_file(h), encoding="utf-8"))
    pool = {norm(d) for l in o["specialist_lists"].values() for d in l}
    miss = [d for d in o["diagnoses"] if norm(d) not in pool]
    print(f"\n=== case {i}: {len(miss)} final items not word-for-word in a specialist list")
    for d in miss:
        print("  FINAL:", d)
    print("  --- specialist items:")
    for s, l in o["specialist_lists"].items():
        print(f"  {s}: " + "; ".join(l))

In [ ]:
import glob
for path in sorted(glob.glob(str(OUT_DIR / "*" / "*.json"))):
    raw = json.load(open(path, encoding="utf-8"))["raw"]
    lines = [l for l in raw.splitlines() if l.strip()]
    extra = [l for l in lines if not LIST_RE.match(l) and l.strip().lower().rstrip(".") != "none"]
    folder = Path(path).parent.name
    print(f"{folder:<12} list_lines={len(lines)-len(extra):>2}  other_lines={len(extra)}")

In [ ]:
ALL = sorted(EV)
run_stage1(ALL)     # ~2.5 h, 232 calls (the 2 pilot cases are skipped)
run_stage2(ALL)     # ~35 min, 58 calls

In [ ]:
import glob, collections

for stage in ("specialists", "moderator"):
    files = glob.glob(str(OUT_DIR / stage / "*.json"))
    outs = [json.load(open(f, encoding="utf-8")) for f in files]
    n = [len(o["diagnoses"]) for o in outs]
    print(f"{stage:<12} files={len(files):>3}  "
          f"finish={dict(collections.Counter(o['finish'] for o in outs))}  "
          f"n_dx min/median/max={min(n)}/{sorted(n)[len(n)//2]}/{max(n)}  "
          f"empty_lists={sum(x == 0 for x in n)}")

missing_mod = [h for h in EV if not mod_file(h).exists()]
print("cases without a moderator file:", len(missing_mod))

In [ ]:
# Step 9: Check the truncated outputs

for stage, cap in (("specialists", SPEC_MAX), ("moderator", FINAL_MAX)):
    for f in sorted(glob.glob(str(OUT_DIR / stage / "*.json"))):
        o = json.load(open(f, encoding="utf-8"))
        if o["finish"] != "length":
            continue
        items = [norm(d) for d in o["diagnoses"]]
        first_rep = next((k + 1 for k, d in enumerate(items) if d in items[:k]), None)
        top = items[:cap]
        print(f"{stage:<12} {o.get('specialist', 'moderator'):<20} listed={len(items):>3} "
              f"first_repeat_at={first_rep}  unique_in_first_{cap}={len(set(top))}/{len(top)}")

hl = {json.load(open(f))["hadm_id"] for f in glob.glob(str(OUT_DIR / "specialists" / "*.json"))
      if json.load(open(f))["finish"] == "length"}
ml = {h for h in EV if json.load(open(mod_file(h)))["finish"] == "length"}
print("moderator-truncated cases that also had a truncated specialist:", len(hl & ml), "of", len(ml))

In [ ]:
# Step 10: Combine into the prediction files, then back up
# holds the moderator's final list for each of the 60 cases.
"""
predictions_model3_27b.json: the official Model 3 output:
* It holds the moderator's final list for each of the 60 cases.
* It has the same keys and fields as Model 1's file (raw, diagnoses, finish, prompt_tokens), so judging.ipynb scores it exactly like Model 1.
* This is the file to compare against Model 1 and Model 2.

specialists_model3_27b.json: the intermediate outputs, not scored as Model 3

For each case, it holds what each of the 4 specialists produced, with the same fields.
It also includes moderator_input: the cleaned lists, capped at 10, that the moderator actually received.
It's only for secondary analyses, such as:
what the moderator kept, merged, dropped or added (e.g. the new-diagnosis rate);
whether the specialists' combined lists already contain the right diagnoses, and whether the moderator improved or lost them.

Both files are built from the per-call files in model3_outputs/, so they don't contain anything new.

"""
p1 = json.load(open("predictions_model1_27b.json", encoding="utf-8"))
FIELDS = ("raw", "diagnoses", "finish", "prompt_tokens")

preds, spec_out = {}, {}
for h, ev in EV.items():
    k = case_key(ev)
    m = json.load(open(mod_file(h), encoding="utf-8"))
    preds[k] = {f: m[f] for f in FIELDS}                      # official Model 3 output
    spec_out[k] = {s: {f: json.load(open(spec_file(h, s), encoding="utf-8"))[f] for f in FIELDS}
                   for s in SPECIALISTS}
    spec_out[k]["moderator_input"] = m["specialist_lists"]    # the capped lists the moderator saw

missing = set(preds) - set(p1)
print("Model 3 cases:", len(preds), "| keys not in Model 1 predictions:", len(missing),
      "| Model 1 cases in file:", len(p1))
assert len(preds) == 60 and not missing

json.dump(preds, open(PRED_FILE, "w", encoding="utf-8"), indent=2)
json.dump(spec_out, open("specialists_model3_27b.json", "w", encoding="utf-8"), indent=2)
print("Saved", PRED_FILE, "and specialists_model3_27b.json")